In [1]:
import pandas as pd
import sqlite3

# Load master CSV into a SQLite database in memory
master = pd.read_csv('data/olist_master.csv')
conn = sqlite3.connect('data/olist.db')
master.to_sql('olist', conn, if_exists='replace', index=False)

print(f"Database created ✓")
print(f"Table 'olist' loaded with {len(master):,} rows")

Database created ✓
Table 'olist' loaded with 113,425 rows


In [2]:
query1 = """
SELECT
    SUBSTR(order_purchase_timestamp, 1, 7) AS month,
    COUNT(DISTINCT order_id)               AS total_orders,
    ROUND(SUM(payment_value), 2)           AS total_revenue,
    ROUND(AVG(payment_value), 2)           AS avg_order_value
FROM olist
WHERE payment_value IS NOT NULL
GROUP BY month
ORDER BY month
"""

result1 = pd.read_sql_query(query1, conn)
print(result1.to_string(index=False))

  month  total_orders  total_revenue  avg_order_value
2016-09             3         388.47            97.12
2016-10           324       76120.17           200.84
2016-12             1          19.62            19.62
2017-01           800      189015.66           195.67
2017-02          1780      349701.93           175.03
2017-03          2682      544738.23           179.13
2017-04          2404      510891.55           189.43
2017-05          3700      731017.09           175.05
2017-06          3245      608891.38           168.62
2017-07          4026      744599.53           162.72
2017-08          4331      876129.37           177.07
2017-09          4285     1023095.49           209.95
2017-10          4631     1031505.53           191.55
2017-11          7544     1599444.18           182.63
2017-12          5673     1057582.34           166.37
2018-01          7269     1415348.54           171.41
2018-02          6728     1311260.71           170.16
2018-03          7211     14

In [3]:
query2 = """
SELECT
    customer_state,
    COUNT(DISTINCT order_id)                                    AS total_orders,
    ROUND(SUM(payment_value), 2)                               AS total_revenue,
    ROUND(SUM(payment_value) * 100.0 / SUM(SUM(payment_value))
          OVER (), 2)                                          AS revenue_pct,
    ROUND(AVG(payment_value), 2)                               AS avg_order_value,
    ROUND(AVG(delivery_days), 1)                               AS avg_delivery_days
FROM olist
WHERE payment_value IS NOT NULL
  AND customer_state IS NOT NULL
GROUP BY customer_state
ORDER BY total_revenue DESC
LIMIT 10
"""

result2 = pd.read_sql_query(query2, conn)
print(result2.to_string(index=False))

customer_state  total_orders  total_revenue  revenue_pct  avg_order_value  avg_delivery_days
            SP         41745     7673188.55        37.48           160.47                8.3
            RJ         12852     2783724.26        13.60           189.77               14.7
            MG         11635     2341861.47        11.44           177.15               11.5
            RS          5466     1152019.17         5.63           183.76               14.7
            PR          5045     1074614.19         5.25           185.69               11.5
            BA          3380      802416.72         3.92           210.00               18.8
            SC          3637      799135.92         3.90           190.23               14.5
            GO          2020      516182.51         2.52           220.03               14.9
            DF          2140      434512.55         2.12           179.48               12.5
            ES          2033      406946.26         1.99           179

In [4]:
query3 = """
SELECT
    CASE WHEN was_late = 1 THEN 'Late' ELSE 'On Time' END AS delivery_status,
    COUNT(DISTINCT order_id)              AS total_orders,
    ROUND(AVG(review_score), 2)           AS avg_review_score,
    ROUND(AVG(delivery_days), 1)          AS avg_delivery_days,
    SUM(CASE WHEN review_score = 1
             THEN 1 ELSE 0 END)           AS one_star_reviews,
    ROUND(SUM(CASE WHEN review_score = 1
             THEN 1 ELSE 0 END) * 100.0 /
             COUNT(DISTINCT order_id), 1) AS pct_one_star
FROM olist
WHERE was_delivered = 1
  AND review_score IS NOT NULL
GROUP BY delivery_status
ORDER BY avg_review_score DESC
"""

result3 = pd.read_sql_query(query3, conn)
print(result3.to_string(index=False))

delivery_status  total_orders  avg_review_score  avg_delivery_days  one_star_reviews  pct_one_star
        On Time         87759              4.21               10.4              8398           9.6
           Late          7613              2.55               30.7              3971          52.2


In [5]:
query4 = """
SELECT
    category_en,
    COUNT(DISTINCT order_id)        AS total_orders,
    ROUND(SUM(payment_value), 2)    AS total_revenue,
    ROUND(AVG(payment_value), 2)    AS avg_order_value,
    ROUND(AVG(review_score), 2)     AS avg_review_score,
    ROUND(AVG(delivery_days), 1)    AS avg_delivery_days
FROM olist
WHERE category_en IS NOT NULL
  AND payment_value IS NOT NULL
GROUP BY category_en
HAVING total_orders >= 100
ORDER BY total_revenue DESC
LIMIT 10
"""

result4 = pd.read_sql_query(query4, conn)
print(result4.to_string(index=False))

          category_en  total_orders  total_revenue  avg_order_value  avg_review_score  avg_delivery_days
       bed_bath_table          9417     1712553.67           154.08              3.90               12.3
        health_beauty          8835     1657373.12           171.45              4.14               11.5
computers_accessories          6689     1585330.45           202.55              3.94               12.8
      furniture_decor          6449     1430176.39           171.61              3.91               12.4
        watches_gifts          5624     1429216.68           238.56              4.02               12.2
       sports_leisure          7720     1392127.56           161.11              4.11               11.7
           housewares          5884     1094758.13           157.20              4.06               10.5
                 auto          3897      852294.33           201.25              4.07               11.8
         garden_tools          3518      838280.75     

In [6]:
query5 = """
SELECT
    seller_id,
    seller_state,
    COUNT(DISTINCT order_id)              AS total_orders,
    ROUND(AVG(review_score), 2)           AS avg_review_score,
    ROUND(AVG(delivery_days), 1)          AS avg_delivery_days,
    SUM(CASE WHEN review_score = 1
             THEN 1 ELSE 0 END)           AS one_star_count,
    ROUND(SUM(CASE WHEN review_score = 1
             THEN 1 ELSE 0 END) * 100.0 /
             COUNT(DISTINCT order_id), 1) AS pct_one_star
FROM olist
WHERE seller_id IS NOT NULL
  AND review_score IS NOT NULL
GROUP BY seller_id
HAVING total_orders >= 50
ORDER BY avg_review_score ASC
LIMIT 10
"""

result5 = pd.read_sql_query(query5, conn)
# Shorten seller_id for display
result5['seller_id'] = result5['seller_id'].str[:8] + '...'
print(result5.to_string(index=False))

  seller_id seller_state  total_orders  avg_review_score  avg_delivery_days  one_star_count  pct_one_star
1ca7077d...           SP           114              2.20               14.5              80          70.2
2eb70248...           SP           198              2.72               17.4              81          40.9
a49928bc...           SP            97              2.93               16.5              37          38.1
54965bbe...           PR            74              2.94               26.4              34          45.9
8e6d7754...           SP            83              2.95               13.5              49          59.0
972d0f9c...           SC            79              2.96               17.9              29          36.7
2a1348e9...           MG            51              3.00               23.1              21          41.2
bbad7e51...           SP            68              3.04               14.8              33          48.5
8444e55c...           MG            97        

In [7]:
conn.close()

print("=" * 55)
print("        SQL ANALYSIS COMPLETE — 5 QUERIES")
print("=" * 55)
print("""
Q1. Monthly revenue trend
    → Nov 2017 peak: R$1.6M, 7,544 orders

Q2. Revenue by state
    → SP = 37.5% of revenue, fastest delivery (8.3 days)
    → BA = highest avg order value (R$210)

Q3. Late delivery impact  ← HEADLINE
    → 52.2% of late orders → 1-star review
    → vs only 9.6% for on-time orders

Q4. Category performance
    → Health & beauty: high revenue + high satisfaction
    → Bed & bath: top revenue but lowest score (3.90)

Q5. Seller risk analysis
    → Worst seller: 2.20 avg, 70.2% one-star rate
    → All bad sellers have delivery days > 13
""")
print("=" * 55)
print("Database connection closed ✓")

        SQL ANALYSIS COMPLETE — 5 QUERIES

Q1. Monthly revenue trend
    → Nov 2017 peak: R$1.6M, 7,544 orders

Q2. Revenue by state
    → SP = 37.5% of revenue, fastest delivery (8.3 days)
    → BA = highest avg order value (R$210)

Q3. Late delivery impact  ← HEADLINE
    → 52.2% of late orders → 1-star review
    → vs only 9.6% for on-time orders

Q4. Category performance
    → Health & beauty: high revenue + high satisfaction
    → Bed & bath: top revenue but lowest score (3.90)

Q5. Seller risk analysis
    → Worst seller: 2.20 avg, 70.2% one-star rate
    → All bad sellers have delivery days > 13

Database connection closed ✓
